In [ ]:
################################################################################################## 
# Ny lambda-funktion som analyserar bilder i S3-bucket med Amazon Rekognition för bilder från Kaggle, och sparar resultatet i en JSON-fil i S3-bucketen.
# Ny bucket: test2pictures
# JSON-fil i mappstrktur här: rekognition_resultat_kaggle_pictures.json (tidigare rekognition_resultat2.json)
################################################################################################## 

import json
import boto3

client = boto3.client('rekognition')
s3 = boto3.client('s3')

true_labels = {
    "buildings1.jpg": ["Building", "City", "Skyline"],
    "buildings2.jpg": ["Building", "City", "Skyline"],
    "buildings3.jpg": ["Building", "Apartment Building", "Architecture"],
    "car1.jpg": ["Car", "Vehicle", "Sedan"],
    "car2.jpg": ["Car", "Vehicle", "Classic Car"],
    "face1.jpg": ["Person", "Face", "Portrait"],
    "face2.jpg": ["Person", "Face", "Portrait"],
    "kitten1.jpg": ["Cat", "Kitten", "Pet"],
    "kitten2.jpg": ["Cat", "Pet"],
    "landscape1.jpg": ["Landscape", "Coast", "Sea"],
    "landscape2.jpg": ["Landscape", "Rock Formation", "Desert"],
    "landscape3.jpg": ["Landscape", "Field", "Grass"],
    "market1.jpg": ["Market", "Street", "People"],
    "market2.jpg": ["Market", "Bazaar", "People"],
    "people1.jpg": ["People", "Person", "Crowd"],
    "people2.jpg": ["People", "Person", "Crowd"],
    "sky1.jpg": ["Sky", "Cloud", "Outdoors"],
    "bird1.jpg": ["Bird", "Parrot", "Animal"],
    "bird2.jpg": ["Bird", "Kingfisher", "Animal"]
}

def lambda_handler(event, context):
    bucket_name = "test2pictures"
    results = []

    s3_response = s3.list_objects_v2(Bucket=bucket_name)

    if "Contents" not in s3_response:
        return {
            'statusCode': 200,
            'body': json.dumps("Inga filer hittades i bucket.", ensure_ascii=False)
        }

    for obj in s3_response['Contents']:
        image_name = obj['Key']
        print(image_name)

        if image_name.startswith('resultat/'):
            continue

        if image_name.endswith('/'):
            continue

        if not image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        rekognition_response = client.detect_labels(
            Image={
                'S3Object': {
                    'Bucket': bucket_name,
                    'Name': image_name
                }
            },
            MaxLabels=10,
            MinConfidence=70
        )

        labels = []
        for label in rekognition_response['Labels']:
            labels.append({
                'Name': label['Name'],
                'Confidence': round(label['Confidence'], 2),
                'Parents': [parent['Name'] for parent in label.get('Parents', [])]
            })

        file_name = image_name.split('/')[-1]

        results.append({
            'image': image_name,
            'file_name': file_name,
            'true_labels': true_labels.get(file_name, []),
            'labels': labels
        })

    json_data = json.dumps(results, ensure_ascii=False, indent=2)

    s3.put_object(
        Bucket=bucket_name,
        Key='resultat/rekognition_resultat2.json',
        Body=json_data.encode('utf-8'),
        ContentType='application/json'
    )

    return {
        'statusCode': 200,
        'body': json.dumps({
            'message': 'Klart',
            'antal_bilder': len(results),
            'json_fil': 'resultat/rekognition_resultat2.json'
        }, ensure_ascii=False)
    }

ModuleNotFoundError: No module named 'boto3'